# Bash Script Check — vodič kroz ceo projekat

Ovaj notebook objašnjava kako projekat klasifikuje Bash skripte kao **safe**, **risky** ili **malicious**. Napravljen je kao edukativni pratilac postojećeg `transformers.ipynb`, ali koristi stvarne module i sačuvani model iz ovog repozitorijuma.

Ciljevi su da razumemo:

1. format i podelu podataka;
2. Byte-level BPE tokenizer;
3. encoder-only Transformer;
4. dve izlazne glave modela;
5. sigurnosna pravila i statističku predikciju;
6. metrike, posebno macro F1;
7. checkpoint serijalizaciju i važne zamke u projektu.

> **Bezbednost:** notebook čita Bash skripte kao običan tekst. Nijedna skripta se ne izvršava.

## 1. Tok podataka

```text
JSONL zapis
    │
    ├── template-aware train/test podela
    │
    ├── Byte-level BPE tokenizer
    │       └── [CLS] + token IDs + dinamički [PAD]
    │
    └── Transformer encoder
            ├── label head  → safe / risky / malicious
            └── reason head → razlog malicious ponašanja

Predikcija zatim kombinuje izlaz modela sa malim brojem pravila visoke preciznosti.
```

In [ ]:
from pathlib import Path
import sys
import json
import torch

# Notebook radi i kada je pokrenut iz root-a i iz transformer-demo foldera.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "bash_classifier").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "bash_classifier").is_dir():
    raise RuntimeError("Nije pronađen root BashScriptCheck projekta")

sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)
print("PyTorch:", torch.__version__)
print("CUDA dostupna:", torch.cuda.is_available())

## 2. Konfiguracija

`ModelConfig` opisuje arhitekturu, dok `TrainingConfig` opisuje podelu podataka i optimizaciju. Dataclass je koristan jer daje imenovana polja, podrazumevane vrednosti i jasniji tip od običnog dictionary-ja.

In [ ]:
from dataclasses import asdict
from bash_classifier.config import LABELS, ModelConfig, TrainingConfig

model_config = ModelConfig()
training_config = TrainingConfig()

print("Labele:", LABELS)
print("Model config:", model_config)
print("Training config:", training_config)
print("Serijalizovan config:", asdict(model_config))

### Zašto `asdict` ovde nije nepotreban?

Checkpoint treba da sadrži bezbedno serijalizovane primitivne vrednosti. Zato je `asdict(config)` opravdan pri čuvanju. Pri učitavanju dictionary mora eksplicitno da se vrati u klasu:

```python
model_config = ModelConfig(**checkpoint["model_config"])
```

Aplikacioni kod pri učitavanju sada radi ovu rekonstrukciju, dok checkpoint i dalje ostaje kompatibilan sa `weights_only=True` učitavanjem.

## 3. Dataset: JSON Lines

Svaki red je samostalan JSON objekat. Najvažnija polja su `label`, `script` i, za malicious uzorke, `category`. Tekst skripte se samo učitava — ne prosleđuje se shell-u.

In [ ]:
from collections import Counter
from bash_classifier.data import load_jsonl

dataset_path = PROJECT_ROOT / "data" / "safe_risky_combined.jsonl"
records = load_jsonl(dataset_path)

print("Broj zapisa:", len(records))
print("Primarne klase:", Counter(row["label"] for row in records))
print("Malicious razlozi:", Counter(
    row.get("category", "") for row in records if row["label"] == "malicious"
))

# Prikazujemo metapodatke, ne izvršavamo script polje.
example = records[0]
print({key: value for key, value in example.items() if key != "script"})
print("Dužina script teksta:", len(example["script"]))

## 4. Podela bez curenja template-a

Obična nasumična podela može poslati dve skoro identične sintetičke skripte i u train i u test skup. `template_fingerprint` normalizuje URL-ove, IP adrese, brojeve i stringove, a `stratified_split` drži jednu familiju u samo jednom skupu.

In [ ]:
from bash_classifier.data import stratified_split, template_fingerprint

train_records, test_records = stratified_split(
    records, training_config.testFraction, training_config.seed
)
train_fingerprints = {template_fingerprint(row["script"]) for row in train_records}
test_fingerprints = {template_fingerprint(row["script"]) for row in test_records}

print("Train zapisi:", len(train_records))
print("Test zapisi:", len(test_records))
print("Presek template familija:", len(train_fingerprints & test_fingerprints))

## 5. Tokenizer

Byte-level BPE ne tretira celu komandu kao jednu reč. Uči česte fragmente bajtova, pa može da obradi nove putanje, opcije, promenljive i adrese. `[CLS]` se dodaje na početak, a `[PAD]` samo do najdužeg primera u trenutnom batch-u.

In [ ]:
from tokenizers import Tokenizer
from bash_classifier.config import CLS_TOKEN, PAD_TOKEN
from bash_classifier.data import BashDataset, collate_batch, required_token_id

checkpoint_path = PROJECT_ROOT / "artifacts" / "bash_transformer.pt"
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
tokenizer = Tokenizer.from_str(checkpoint["tokenizer_json"])

sample_text = "#!/usr/bin/env bash\nprintf '%s\\n' hello"
encoding = tokenizer.encode(sample_text)
print("Veličina rečnika:", tokenizer.get_vocab_size())
print("Tokeni:", encoding.tokens[:30])
print("ID-jevi:", encoding.ids[:30])
print("CLS ID:", required_token_id(tokenizer, CLS_TOKEN))
print("PAD ID:", required_token_id(tokenizer, PAD_TOKEN))

In [ ]:
reason_names = tuple(checkpoint["reason_names"])
reason_to_id = {name: index for index, name in enumerate(reason_names)}
small_dataset = BashDataset(
    train_records[:3], tokenizer, model_config.contextSize, reason_to_id
)
batch = collate_batch(
    [small_dataset[i] for i in range(3)],
    padId=required_token_id(tokenizer, PAD_TOKEN),
)

for name, tensor in batch.items():
    print(f"{name:15s} shape={tuple(tensor.shape)} dtype={tensor.dtype}")

## 6. Model

Model je encoder-only Transformer:

1. token ID → embedding;
2. dodavanje sinusoidalnog positional encoding-a;
3. više `TransformerEncoderLayer` blokova;
4. mean pooling nemaskiranih pozicija;
5. `labelHead` daje tri logita;
6. `reasonHead` daje logite malicious razloga.

Logiti nisu verovatnoće. `softmax` ih pretvara u raspodelu čiji je zbir 1.

In [ ]:
from bash_classifier.model import make_model

# Eksplicitna rekonstrukcija ovde pokazuje šta load_checkpoint radi interno.
saved_model_config = ModelConfig(**checkpoint["model_config"])
saved_training_config = TrainingConfig(**checkpoint["training_config"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = make_model(tokenizer, reason_names, saved_model_config, device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

parameter_count = sum(parameter.numel() for parameter in model.parameters())
print("Uređaj:", device)
print("Broj parametara:", f"{parameter_count:,}")
print(model)

In [ ]:
from bash_classifier.data import move_batch_to_device

device_batch = move_batch_to_device(batch, device)
with torch.inference_mode():
    label_logits, reason_logits = model(
        device_batch["input_ids"], device_batch["attention_mask"]
    )

print("Label logits shape:", tuple(label_logits.shape))
print("Reason logits shape:", tuple(reason_logits.shape))
print("Prva raspodela:", dict(zip(
    LABELS, label_logits.softmax(dim=1)[0].cpu().tolist()
)))

## 7. Security rules + model

Pravila služe samo za jasne obrasce, kao što su eksplicitni reverse shell, upload osetljivog fajla, promena firewall-a ili očigledna beskonačna busy petlja. Ako nema pravila, koristi se model. Ako je confidence ispod praga, rezultat postaje `uncertain`.

Važno: broj `0.99` koji vraća ručno pravilo nije statistički kalibrisana verovatnoća; to je fiksna vrednost prioriteta pravila.

In [ ]:
from bash_classifier.security_rules import find_security_rule

rule_examples = {
    "read_only": "dig example.com A\ndig example.com MX",
    "firewall": "sudo ufw disable",
    "busy_loop": "while true; do echo working; done",
}

for name, script_text in rule_examples.items():
    print(name, "->", find_security_rule(script_text))

## 8. Evaluacija postojećeg checkpoint-a — bez treninga

Sledeća ćelija samo rekonstruiše isti held-out split pomoću seed-a sačuvanog u checkpoint-u i izvršava forward pass. Ne poziva optimizer, `backward()` niti bilo koju training funkciju.

In [ ]:
from bash_classifier.data import make_data_loader
from bash_classifier.evaluation import print_metrics, test_model

checkpoint_train_records, checkpoint_test_records = stratified_split(
    records,
    saved_training_config.testFraction,
    saved_training_config.seed,
)
test_dataset = BashDataset(
    checkpoint_test_records,
    tokenizer,
    saved_model_config.contextSize,
    reason_to_id,
)
test_loader = make_data_loader(
    test_dataset,
    tokenizer,
    saved_training_config.batchSize,
    device,
)

metrics = test_model(model, test_loader, device, reason_names)
print("Held-out samples:", len(test_dataset))
print_metrics(metrics)

### Kako se čita F1?

$$F1 = 2 \cdot \frac{precision \cdot recall}{precision + recall}$$

- **precision** pita: od svega što je model označio ovom klasom, koliko je tačno?
- **recall** pita: od svih stvarnih primera ove klase, koliko ih je model pronašao?
- **macro F1** računa F1 svake klase zasebno, pa uzima običan prosek. Zato velika klasa ne može da sakrije loš rezultat male klase.

Za postojeći checkpoint, izmeren macro F1 je približno **0.9986** na 811 sintetičkih held-out primera. To nije dokaz iste pouzdanosti na realnim skriptama iz drugih izvora.

## 9. Kako izgleda trening — bez pokretanja

Trening tok u `training.py` radi sledeće:

1. postavlja seed;
2. deli podatke i trenira tokenizer samo na train skupu;
3. pravi class weights zbog neuravnoteženih klasa;
4. računa label loss i reason loss;
5. izvršava backpropagation, gradient clipping i AdamW korak;
6. posle svake epohe čuva model, optimizer, scaler, loss istoriju, broj završenih epoha, konfiguracije i tokenizer;
7. `--resume --epochs N` nastavlja novi checkpoint do ukupno `N` epoha.

Notebook namerno nema poziv `train_new_model`, `train_model`, `backward` ili `optimizer.step`, pa njegovo izvršavanje neće trenirati niti menjati checkpoint.

## 10. Najvažnije lekcije i vežbe

1. Uporedi accuracy i macro F1 i objasni zašto nisu identični.
2. Pronađi jedinu grešku u confusion matrix-i postojećeg checkpoint-a.
3. Probaj različite bezazlene Bash tekstove kroz tokenizer i posmatraj BPE fragmente.
4. Proveri kako `template_fingerprint` menja URL, IP, broj i quoted string.
5. Razmisli da li model treba da koristi mean pooling ili samo `[CLS]` reprezentaciju.
6. Razdvoji fiksni prioritet security pravila od kalibrisane verovatnoće modela.
7. Pre produkcione upotrebe napravi nezavisan, ručno pregledan real-world test skup.

> Klasifikator je pomoć pri trijaži, ne zamena za sandbox, statičku analizu, pregled čoveka ili druge sigurnosne kontrole.